Database Connection

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Import tables from database to DataFrame

In [5]:
import pandas as pd
print("⏳ Άντληση μοναδικών ονομάτων ομάδων από την PostgreSQL...")

# 1. Παίρνουμε όλες τις ομάδες
query_us = """
SELECT DISTINCT team_name FROM (
    SELECT home_team AS team_name FROM understat_matches
    UNION
    SELECT away_team AS team_name FROM understat_matches
) AS us_teams
"""
query_sb = """
SELECT DISTINCT team_name FROM (
    SELECT home_team AS team_name FROM statsbomb_matches
    UNION
    SELECT away_team AS team_name FROM statsbomb_matches
) AS sb_teams
"""

df_us_teams = pd.read_sql_query(query_us, engine)
df_sb_teams = pd.read_sql_query(query_sb, engine)

us_teams_list = df_us_teams['team_name'].dropna().tolist()
sb_teams_list = df_sb_teams['team_name'].dropna().tolist()

print(f"Βρέθηκαν {len(us_teams_list)} ομάδες στο Understat και {len(sb_teams_list)} ομάδες στο StatsBomb.")

⏳ Άντληση μοναδικών ονομάτων ομάδων από την PostgreSQL...
Βρέθηκαν 200 ομάδες στο Understat και 213 ομάδες στο StatsBomb.


Fuzzy Team Name Matching

In [6]:
from rapidfuzz import process, fuzz

# Advanced Fuzzy Matching με RapidFuzz
print("⏳ Προσπάθεια αυτόματου ταιριάσματος με RapidFuzz (token_set_ratio)...")
mapping_data = []

for sb_team in sb_teams_list:
    # Χρησιμοποιούμε token_set_ratio και ζητάμε τουλάχιστον 60% σκορ
    result = process.extractOne(sb_team, us_teams_list, scorer=fuzz.token_set_ratio, score_cutoff=60)

    if result:
        us_match = result[0]  # Το όνομα που βρέθηκε
        score = result[1]     # Το ποσοστό βεβαιότητας (0 - 100)
    else:
        us_match = None
        score = 0

    mapping_data.append({
        'statsbomb_team': sb_team,
        'understat_team': us_match,
        'confidence_score': round(score, 1)
    })

df_team_mapping = pd.DataFrame(mapping_data)

# Ταξινομούμε αύξουσα βάσει score. Έτσι τα πιο "αδύναμα" ταιριάσματα θα είναι στην κορυφή του CSV για να τα ελέγξεις πρώτα!
df_team_mapping = df_team_mapping.sort_values(by='confidence_score')

# 3. Εξαγωγή σε CSV για τον τελικό χειροκίνητο έλεγχο
#output_file = 'team_mapping_review.csv'
#df_team_mapping.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✅ Το λεξικό ονομάτων δημιουργήθηκε")
print("Δείγμα από τα πιο 'δύσκολα' ταιριάσματα (με το χαμηλότερο σκορ):")
display(df_team_mapping.head(15))

⏳ Προσπάθεια αυτόματου ταιριάσματος με RapidFuzz (token_set_ratio)...

✅ Το λεξικό ονομάτων δημιουργήθηκε
Δείγμα από τα πιο 'δύσκολα' ταιριάσματα (με το χαμηλότερο σκορ):


,statsbomb_team,understat_team,confidence_score
10,Sweden,NaN,0.0
14,Mohun Bagan Super Giant,NaN,0.0
9,Czech Republic,NaN,0.0
28,Saudi Arabia,NaN,0.0
25,Wales,NaN,0.0
27,Ukraine,NaN,0.0
20,Finland,NaN,0.0
22,Portugal,NaN,0.0
53,Nigeria,NaN,0.0
56,Congo DR,NaN,0.0


Import team mapping to Database

In [7]:
# Κρατάμε ΜΟΝΟ όσα έχουν σκορ 80 και πάνω
df_mapping = df_team_mapping[df_team_mapping['confidence_score'] >= 80].copy()
# Κρατάμε μόνο τις 2 στήλες των ονομάτων, δεν χρειαζόμαστε το σκορ πια
df_mapping = df_mapping[['statsbomb_team', 'understat_team']]
print(f"Μετά το φιλτράρισμα έμειναν {len(df_mapping)} έγκυρες αντιστοιχίσεις.")
table_name = 'mapping_team_names'
try:
    with engine.begin() as conn:

        df_mapping.to_sql(
            name=table_name,
            con=conn,
            if_exists='replace',
            index=False
        )
    print(f"✅ ΕΠΙΤΥΧΙΑ! Ο πίνακας '{table_name}' ανέβηκε στη βάση δεδομένων")
except Exception as e:
    print(f"❌ Σφάλμα: {e}")

Μετά το φιλτράρισμα έμειναν 116 έγκυρες αντιστοιχίσεις.
✅ ΕΠΙΤΥΧΙΑ! Ο πίνακας 'team_mapping' ανέβηκε στη βάση δεδομένων


Dataframe με ΟΛΑ τα κοινα matches

In [16]:
query_overlaps = """
SELECT
    sb.match_id AS statsbomb_match_id,
    us.match_id AS understat_match_id,
    sb.match_date AS statsbomb_date,
    us.match_date AS understat_date,
    sb.home_team AS statsbomb_home_name,
    us.home_team AS understat_home_name,
    sb.away_team AS statsbomb_away_name,
    us.away_team AS understat_away_name
FROM statsbomb_matches sb
-- 1. Μετάφραση ονόματος Γηπεδούχου
JOIN mapping_team_names map_home ON sb.home_team = map_home.statsbomb_team
-- 2. Μετάφραση ονόματος Φιλοξενούμενου
JOIN mapping_team_names map_away ON sb.away_team = map_away.statsbomb_team
-- 3. Εύρεση του αντίστοιχου αγώνα στο Understat
JOIN understat_matches us
  ON us.home_team = map_home.understat_team
  AND us.away_team = map_away.understat_team
  AND DATE(sb.match_date) = DATE(us.match_date);
"""
print("⏳ Ανίχνευση επικαλύψεων στη Βάση Δεδομένων...")
# Εκτέλεση του query και αποθήκευση στο DataFrame
df_overlapping = pd.read_sql_query(query_overlaps, engine)
print(f"✅ Ολοκληρώθηκε! Βρέθηκαν {len(df_overlapping)} αγώνες που υπάρχουν και στα δύο Datasets.")
display(df_overlapping.head(10))


⏳ Ανίχνευση επικαλύψεων στη Βάση Δεδομένων...
✅ Ολοκληρώθηκε! Βρέθηκαν 1576 αγώνες που υπάρχουν και στα δύο Datasets.


,statsbomb_match_id,understat_match_id,statsbomb_date,understat_date,statsbomb_home_name,understat_home_name,statsbomb_away_name,understat_away_name
0,3879869,923,2016-05-14,2016-05-14 22:45:00,AC Milan,AC Milan,AS Roma,Roma
1,3879607,662,2015-11-07,2015-11-07 23:45:00,AC Milan,AC Milan,Atalanta,Atalanta
2,3879674,727,2016-01-06,2016-01-06 18:00:00,AC Milan,AC Milan,Bologna,Bologna
3,3879836,890,2016-04-21,2016-04-21 22:45:00,AC Milan,AC Milan,Carpi,Carpi
4,3879594,644,2015-10-28,2015-10-28 23:45:00,AC Milan,AC Milan,Chievo,Chievo
5,3878551,562,2015-08-29,2015-08-29 22:45:00,AC Milan,AC Milan,Empoli,Empoli
6,3879690,749,2016-01-17,2016-01-17 23:45:00,AC Milan,AC Milan,Fiorentina,Fiorentina
7,3879851,906,2016-05-01,2016-05-01 17:00:00,AC Milan,AC Milan,Frosinone,Frosinone
8,3879742,796,2016-02-14,2016-02-14 15:30:00,AC Milan,AC Milan,Genoa,Genoa
9,3879652,704,2015-12-13,2015-12-13 18:00:00,AC Milan,AC Milan,Hellas Verona,Verona


Import overlapping matches to DB

In [18]:
from sqlalchemy import text

table_name_overlaps = 'mapping_overlapping_matches'

# Κρατάμε ΜΟΝΟ τα IDs από το DataFrame που είχαμε φτιάξει πριν
df_overlapping_minimal = df_overlapping[['statsbomb_match_id', 'understat_match_id']].copy()

# Το νέο, απόλυτα Κανονικοποιημένο (Normalized) Schema
create_table_sql_overlaps = f"""
DROP TABLE IF EXISTS public.{table_name_overlaps};
CREATE TABLE public.{table_name_overlaps}
(
    statsbomb_match_id INTEGER,
    understat_match_id INTEGER,
    PRIMARY KEY (statsbomb_match_id, understat_match_id)
);
"""

print(f"⏳ Δημιουργία βελτιστοποιημένου πίνακα και εισαγωγή {len(df_overlapping_minimal)} εγγραφών...")

try:
    with engine.begin() as conn:
        # 1. Δημιουργία
        conn.execute(text(create_table_sql_overlaps))

        # 2. Ανέβασμα (χρησιμοποιούμε το minimal DataFrame πλέον)
        df_overlapping_minimal.to_sql(
            name=table_name_overlaps,
            con=conn,
            if_exists='append',
            index=False,
            method='multi'
        )

    print(f"✅ ΕΠΙΤΥΧΙΑ! Ο πίνακας '{table_name_overlaps}' αποθηκεύτηκε στη Βάση")

except Exception as e:
    print(f"❌ Σφάλμα κατά την εισαγωγή: {e}")

⏳ Δημιουργία βελτιστοποιημένου πίνακα και εισαγωγή 1576 εγγραφών...
✅ ΕΠΙΤΥΧΙΑ! Ο 'ελαφρύς' πίνακας 'mapping_overlapping_matches' αποθηκεύτηκε στη Βάση
